# Fine-tune IndicBART for Hindi / Marathi Summarization (Kaggle)

This notebook fine-tunes [`ai4bharat/IndicBARTSS`](https://huggingface.co/ai4bharat/IndicBARTSS) on a sample of [XL-Sum](https://huggingface.co/datasets/csebuetnlp/xlsum) for **Hindi** or **Marathi**. XL-Sum provides news article → summary pairs in 44 languages.

### How to run on Kaggle
1. Create a new Notebook at [kaggle.com](https://www.kaggle.com).
2. **Settings → Accelerator → GPU T4 x2** (any GPU works).
3. **Settings → Internet → On** (needed to download models/data).
4. Run for **Hindi** first. Then set `LANG = "marathi"` (and `LANG_CODE = "mr_IN"`) and **Run All** again to train the Marathi model.

The weights are saved under `./indicbart-finetuned-hindi` and `./indicbart-finetuned-marathi`. Place them in `models/hindi/` and `models/marathi/` in the repo.

In [ ]:
# Install training dependencies.
!pip install -q transformers datasets evaluate rouge_score nltk accelerate sentencepiece


In [ ]:
import nltk
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

nltk.download("punkt")

print("CUDA available:", torch.cuda.is_available())

# ===== EDIT ME =====
LANG = "hindi"          # "hindi" or "marathi"
LANG_CODE = "hi_IN"     # "mr_IN" for Marathi
# ===================

MODEL_ID = "ai4bharat/IndicBARTSS"
DATASET = "csebuetnlp/xlsum"

# Small subset for a fast demo run; remove .select() to use the full data.
TRAIN_SIZE = 2000
EVAL_SIZE = 200

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


In [ ]:
# XL-Sum exposes one config per language, so we load only what we need.
dataset = load_dataset(DATASET, LANG)

train = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
valid = dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))
test = dataset["test"].shuffle(seed=42).select(range(EVAL_SIZE))

print("Article :", train[0]["text"][:200])
print("---")
print("Summary :", train[0]["summary"][:200])


In [ ]:
# IndicBARTSS uses a SentencePiece tokenizer; fast tokenizer is unavailable.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, do_lower_case=False, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)


In [ ]:
def get_decoder_start_id(tokenizer, lang_code):
    """Resolve the forced decoder-start token id (mBART-style or <2xx>-style)."""
    lang_to_id = getattr(tokenizer, "lang_code_to_id", None) or {}
    if lang_code in lang_to_id:
        return int(lang_to_id[lang_code])
    token_id = tokenizer.convert_tokens_to_ids(f"<2{lang_code[:2]}>")
    if token_id != tokenizer.unk_token_id:
        return int(token_id)
    return tokenizer.bos_token_id


# Pin the decoder language so eval-time generation emits the right script.
model.generation_config.forced_bos_token_id = get_decoder_start_id(tokenizer, LANG_CODE)
print("forced_bos_token_id =", model.generation_config.forced_bos_token_id)


In [ ]:
def preprocess_function(examples):
    inputs = examples["text"]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train.map(preprocess_function, batched=True, remove_columns=train.column_names)
tokenized_valid = valid.map(preprocess_function, batched=True, remove_columns=valid.column_names)


In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result.items()}
    result["gen_len"] = round(
        float(np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in predictions])), 2
    )
    return result


In [ ]:
output_dir = f"./indicbart-finetuned-{LANG}"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


In [ ]:
metrics = trainer.evaluate(eval_dataset=tokenized_valid, max_length=MAX_TARGET_LENGTH, num_beams=4)
print(metrics)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to", output_dir)


In [ ]:
# Try it on a held-out example.
article = test[0]["text"]
inputs = tokenizer(article, max_length=MAX_INPUT_LENGTH, truncation=True, return_tensors="pt").to(model.device)
summary_ids = model.generate(**inputs, max_length=128, min_length=20, num_beams=4, early_stopping=True)
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


### Train the second language
Change `LANG` to `"marathi"` and `LANG_CODE` to `"mr_IN"` in the **EDIT ME** cell, then **Run All** again. The Marathi weights are saved to `./indicbart-finetuned-marathi`.

### Use it in the app
1. Download both folders from Kaggle → **Output** tab.
2. Place them at `models/hindi/` and `models/marathi/` in the repo.
3. The app auto-detects them and uses them instead of the public `ai4bharat/IndicBARTSS`.